# Average vs. Conflict-Aware Group Ranking

This notebook isolates the ranking step from the future LightGCN model. It assumes that each candidate movie already has a normalized preference score for user A (`qA`) and user B (`qB`).

## 1. Understand the formula

The shared score is

$$S(i) = (1 - \lambda) \frac{q_A(i) + q_B(i)}{2} + \lambda \min(q_A(i), q_B(i)).$$

- `lambda = 0`: average-score baseline.
- `lambda = 1`: least-misery ranking based on the worse-off member.
- Values between zero and one balance average and minimum satisfaction.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from group_movie_recommender.algorithms.group_ranking import (
    normalize_member_scores,
    rank_group_candidates,
    score_group_candidates,
)
from group_movie_recommender.evaluation.metrics import evaluate_shared_rankings

## 2. Inspect three candidate movies

Movie 101 strongly favors A, movie 103 strongly favors B, and movie 102 is a compromise.

In [ ]:
candidate_scores = pd.DataFrame(
    {
        "movieId": [101, 102, 103],
        "qA": [1.0, 0.6, 0.3],
        "qB": [0.4, 0.6, 0.9],
    }
)
candidate_scores

## 3. Compare the group scores

With the average baseline, movie 101 wins because its high score for A offsets its lower score for B. Increasing lambda penalizes this imbalance.

In [ ]:
average_scores = score_group_candidates(
    candidate_scores,
    conflict_weight=0.0,
)
conflict_scores = score_group_candidates(
    candidate_scores,
    conflict_weight=0.75,
)

comparison = average_scores[["movieId", "averageScore", "minimumScore"]].copy()
comparison["score_lambda_0"] = average_scores["groupScore"]
comparison["score_lambda_075"] = conflict_scores["groupScore"]
comparison

In [ ]:
average_ranking = rank_group_candidates(
    candidate_scores,
    conflict_weight=0.0,
    k=3,
)
conflict_ranking = rank_group_candidates(
    candidate_scores,
    conflict_weight=0.75,
    k=3,
)

display(average_ranking[["rank", "movieId", "groupScore"]])
display(conflict_ranking[["rank", "movieId", "groupScore"]])

The average method ranks movie 101 first. The conflict-aware method ranks the balanced movie 102 first.

## 4. Connect ranking to the offline metrics

Assume A's relevant movies are 101 and 102, while B's relevant movies are 102 and 103. At rank one, movie 102 satisfies both members.

In [ ]:
def add_pair_columns(ranking):
    result = ranking[["rank", "movieId"]].copy()
    result["pairId"] = 0
    result["userA"] = 1
    result["userB"] = 2
    return result

relevant_items = {1: {101, 102}, 2: {102, 103}}
_, average_metrics = evaluate_shared_rankings(
    add_pair_columns(average_ranking),
    relevant_items,
    catalog_size=3,
    k=1,
)
_, conflict_metrics = evaluate_shared_rankings(
    add_pair_columns(conflict_ranking),
    relevant_items,
    catalog_size=3,
    k=1,
)

pd.DataFrame([average_metrics, conflict_metrics], index=["average", "conflict-aware"])

## 5. Normalize future model scores

LightGCN may produce different raw score ranges for A and B. Percentile normalization makes the two members comparable before aggregation.

In [ ]:
raw_model_scores = pd.DataFrame(
    {
        "movieId": [101, 102, 103],
        "scoreA": [8.0, 3.0, -1.0],
        "scoreB": [-2.0, 0.5, 5.0],
    }
)
normalize_member_scores(raw_model_scores)